# Target Encoding — Origin & Dest

Computes target encoding for `Origin` and `Dest` using training data only,
then joins the encoded columns to all three splits and saves the results.

**Input:** `flights_split/train_2018_2022.parquet`, `validate_2023.parquet`, `test_2024.parquet`  
**Output:** `flights_encoded/train_encoded.parquet`, `validate_encoded.parquet`, `test_encoded.parquet`  
**New columns added:** `origin_delay_rate`, `dest_delay_rate`

---
## 1. Setup

In [1]:
import pandas as pd
import numpy as np
import os

print('All imports successful!')

All imports successful!


---
## 2. Define Paths

In [2]:
BASE_PATH   = 'flights_split'
OUTPUT_PATH = 'flights_encoded'

TRAIN_PATH    = f'{BASE_PATH}/train_2018_2022.parquet'
VALIDATE_PATH = f'{BASE_PATH}/validate_2023.parquet'
TEST_PATH     = f'{BASE_PATH}/test_2024.parquet'

TRAIN_OUT    = f'{OUTPUT_PATH}/train_encoded.parquet'
VALIDATE_OUT = f'{OUTPUT_PATH}/validate_encoded.parquet'
TEST_OUT     = f'{OUTPUT_PATH}/test_encoded.parquet'

---
## 3. Load Data

In [3]:
train_df    = pd.read_parquet(TRAIN_PATH)
validate_df = pd.read_parquet(VALIDATE_PATH)
test_df     = pd.read_parquet(TEST_PATH)

print(f'Train    : {train_df.shape[0]:>10,} rows x {train_df.shape[1]} cols')
print(f'Validate : {validate_df.shape[0]:>10,} rows x {validate_df.shape[1]} cols')
print(f'Test     : {test_df.shape[0]:>10,} rows x {test_df.shape[1]} cols')

Train    : 31,149,502 rows x 56 cols
Validate :  6,743,403 rows x 56 cols
Test     :  6,965,246 rows x 56 cols


---
## 4. Compute Target Encoding from Training Data Only

For each airport code, compute the mean of `ArrDel15` using only training rows.
This gives the historical delay rate for that airport — what fraction of flights
were delayed ≥15 min during 2018–2022.

In [4]:
# Global training delay rate — used as fallback for unseen categories
global_delay_rate = train_df['ArrDel15'].mean()
print(f'Global training delay rate: {global_delay_rate:.4f}')

Global training delay rate: 0.1785


In [5]:
# Origin encoding
origin_encoding = (
    train_df.groupby('Origin')['ArrDel15']
    .mean()
    .reset_index()
    .rename(columns={'ArrDel15': 'origin_delay_rate'})
)

print(f'Origin encoding: {len(origin_encoding)} airports')
print(origin_encoding.sort_values('origin_delay_rate', ascending=False).head(10))

Origin encoding: 382 airports
    Origin  origin_delay_rate
65     CDB           0.525000
380    YNG           0.500000
178    ILG           0.405063
9      ADK           0.385246
286    PPG           0.344411
241    MMH           0.328662
322    SCK           0.316870
255    OGD           0.316629
208    LCK           0.314840
177    IFP           0.311111


In [6]:
# Dest encoding
dest_encoding = (
    train_df.groupby('Dest')['ArrDel15']
    .mean()
    .reset_index()
    .rename(columns={'ArrDel15': 'dest_delay_rate'})
)

print(f'Dest encoding: {len(dest_encoding)} airports')
print(dest_encoding.sort_values('dest_delay_rate', ascending=False).head(10))

Dest encoding: 382 airports
    Dest  dest_delay_rate
380  YNG         1.000000
65   CDB         0.375000
52   BQN         0.330469
39   BIH         0.312321
289  PSE         0.306577
273  PGD         0.298092
286  PPG         0.296073
274  PGV         0.282540
325  SFB         0.281199
29   AZA         0.280922


---
## 5. Join Encodings to All Three Splits

The same lookup tables computed from training are joined to train, validate, and test.
Any airport that appears in validate/test but not in training gets filled with
the global training delay rate.

In [7]:
def apply_encoding(df, origin_enc, dest_enc, global_rate):
    df = df.merge(origin_enc, on='Origin', how='left')
    df = df.merge(dest_enc, on='Dest', how='left')
    df['origin_delay_rate'] = df['origin_delay_rate'].fillna(global_rate)
    df['dest_delay_rate']   = df['dest_delay_rate'].fillna(global_rate)
    return df

train_df    = apply_encoding(train_df,    origin_encoding, dest_encoding, global_delay_rate)
validate_df = apply_encoding(validate_df, origin_encoding, dest_encoding, global_delay_rate)
test_df     = apply_encoding(test_df,     origin_encoding, dest_encoding, global_delay_rate)

print('Encoding applied to all three splits.')

Encoding applied to all three splits.


---
## 6. Verify

In [8]:
# 6.1 Null check — no nulls expected after fillna
for name, df in [('Train', train_df), ('Validate', validate_df), ('Test', test_df)]:
    nulls = df[['origin_delay_rate', 'dest_delay_rate']].isnull().sum()
    status = '✅' if nulls.sum() == 0 else '⚠️'
    print(f'{status} {name}: origin_delay_rate nulls={nulls["origin_delay_rate"]}, dest_delay_rate nulls={nulls["dest_delay_rate"]}')

✅ Train: origin_delay_rate nulls=0, dest_delay_rate nulls=0
✅ Validate: origin_delay_rate nulls=0, dest_delay_rate nulls=0
✅ Test: origin_delay_rate nulls=0, dest_delay_rate nulls=0


In [9]:
# 6.2 Range check — values should be between 0 and 1
for col in ['origin_delay_rate', 'dest_delay_rate']:
    mn   = train_df[col].min()
    mx   = train_df[col].max()
    mean = train_df[col].mean()
    print(f'{col}: min={mn:.4f}  max={mx:.4f}  mean={mean:.4f}')

origin_delay_rate: min=0.0681  max=0.5250  mean=0.1785
dest_delay_rate: min=0.0695  max=1.0000  mean=0.1785


In [11]:
# 6.3 Final shape confirmation
for name, df in [('Train', train_df), ('Validate', validate_df), ('Test', test_df)]:
    print(f'{name}: {df.shape[0]:>10,} rows x {df.shape[1]} cols')

Train: 31,149,502 rows x 58 cols
Validate:  6,743,403 rows x 58 cols
Test:  6,965,246 rows x 58 cols


# 6.3 Final shape confirmation
for name, df in [('Train', train_df), ('Validate', validate_df), ('Test', test_df)]:
    print(f'{name}: {df.shape[0]:>10,} rows x {df.shape[1]} cols')

In [12]:
os.makedirs(OUTPUT_PATH, exist_ok=True)

train_df.to_parquet(TRAIN_OUT,    index=False)
validate_df.to_parquet(VALIDATE_OUT, index=False)
test_df.to_parquet(TEST_OUT,     index=False)

print('Saved:')
print(f'  {TRAIN_OUT}')
print(f'  {VALIDATE_OUT}')
print(f'  {TEST_OUT}')

Saved:
  flights_encoded/train_encoded.parquet
  flights_encoded/validate_encoded.parquet
  flights_encoded/test_encoded.parquet


In [13]:
train_df.head()

,Year,Quarter,Month,DayofMonth,DayOfWeek,FlightDate,Reporting_Airline,Flight_Number_Reporting_Airline,Origin,Dest,...,dest_is_fog,dest_low_visibility,dest_high_wind,dest_severe_weather,is_weekend,is_holiday,origin_weather_missing,dest_weather_missing,origin_delay_rate,dest_delay_rate
0,2018,1,1,1,1,1514764800000000000,WN,1491.0,ABQ,BWI,...,0,0,0,0,0,1,0,0,0.157754,0.163498
1,2018,1,1,1,1,1514764800000000000,UA,2102.0,IAH,LAS,...,0,0,0,0,0,1,0,0,0.177760,0.189581
2,2018,1,1,1,1,1514764800000000000,UA,2101.0,IAH,SFO,...,0,0,0,0,0,1,0,0,0.177760,0.203962
3,2018,1,1,1,1,1514764800000000000,UA,2100.0,IAH,FLL,...,0,0,0,0,0,1,0,0,0.177760,0.210845
4,2018,1,1,1,1,1514764800000000000,UA,2099.0,SAN,IAH,...,0,0,0,0,0,1,0,0,0.155637,0.171826
